# AIML Practical Assignment: Ensemble Learning
## Stacking, AdaBoost, and Majority Voting

**Dataset:** Kaggle's Breast Cancer Wisconsin (Diagnostic) dataset (569 samples, 30 numerical features after removing the ID and empty column). The notebook downloads the public Kaggle dataset directly; no Kaggle API token is required.

**Objectives:** implement base classifiers, stacking with a meta-classifier, AdaBoost with a chosen weak learner, and majority voting; compare performance using classification metrics, confusion matrices, reports, and graphs.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, log_loss,
    classification_report, confusion_matrix
)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier, AdaBoostClassifier, VotingClassifier

RANDOM_STATE = 42

## 2. Load built-in dataset

In [ ]:
import pandas as pd

!wget -q https://www.kaggle.com/api/v1/datasets/download/uciml/breast-cancer-wisconsin-data -O breast_cancer.zip
!unzip -o -q breast_cancer.zip

df = pd.read_csv("data.csv")
df = df.drop(columns=["id", "Unnamed: 32"], errors="ignore")
df["diagnosis"] = df["diagnosis"].map({"B": 0, "M": 1})

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"].astype(int)

print("Dataset shape:", X.shape)
print("Target counts:")
print(y.map({0: "Benign", 1: "Malignant"}).value_counts())
display(df.head())


## 3. Split data and define base classifiers

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

base_estimators = [
    ("svm", make_pipeline(StandardScaler(), SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))),
    ("decision_tree", DecisionTreeClassifier(max_depth=5, criterion="entropy", random_state=RANDOM_STATE)),
    ("knn", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))),
    ("gaussian_nb", GaussianNB())
]

base_models = {
    "SVM": make_pipeline(StandardScaler(), SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, criterion="entropy", random_state=RANDOM_STATE),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Gaussian Naive Bayes": GaussianNB()
}

for name, model in base_models.items():
    model.fit(X_train, y_train)
    print(name, "trained successfully")

## 4. Stacking ensemble

In [ ]:
# The base classifiers generate predictions; Logistic Regression learns how to combine them.
stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    stack_method="predict_proba",
    cv=5,
    passthrough=False
)
stacking_model.fit(X_train, y_train)
print("Stacking ensemble trained successfully")

## 5. AdaBoost ensemble

In [ ]:
# A decision stump is a weak learner. AdaBoost focuses more on previously misclassified samples.
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
    n_estimators=100,
    learning_rate=0.5,
    random_state=RANDOM_STATE
)
ada_model.fit(X_train, y_train)
print("AdaBoost ensemble trained successfully")

## 6. Majority Voting ensemble

In [ ]:
# Hard voting predicts the class receiving the most votes from base classifiers.
voting_model = VotingClassifier(estimators=base_estimators, voting="hard")
voting_model.fit(X_train, y_train)
print("Majority Voting ensemble trained successfully")

## 7. Evaluate base and ensemble models

In [ ]:
models = {**base_models, "Stacking": stacking_model, "AdaBoost": ada_model, "Voting (Hard)": voting_model}
predictions, probabilities, rows = {}, {}, []

for name, model in models.items():
    pred = model.predict(X_test)
    predictions[name] = pred

    if name == "Voting (Hard)":
        # Estimate class probabilities from the fraction of base estimators voting for class 1.
        member_preds = np.column_stack([est.predict(X_test) for est in model.estimators_])
        p1 = member_preds.mean(axis=1)
        prob = np.column_stack([1-p1, p1])
    else:
        prob = model.predict_proba(X_test)
    probabilities[name] = prob

    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-Score": f1_score(y_test, pred, zero_division=0),
        "Log Loss": log_loss(y_test, np.clip(prob, 1e-15, 1-1e-15), labels=[0,1])
    })

results_df = pd.DataFrame(rows).sort_values("F1-Score", ascending=False).reset_index(drop=True)
display(results_df.round(4))

## 8. Confusion matrices for every model

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
for ax, (name, pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, pred, labels=[0,1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")
    ax.set_xticklabels(["Benign", "Malignant"])
    ax.set_yticklabels(["Benign", "Malignant"], rotation=0)
for ax in axes[len(predictions):]:
    ax.axis("off")
plt.suptitle("Confusion Matrices: Base and Ensemble Models")
plt.tight_layout()
plt.show()

## 9. Classification reports

In [ ]:
for name, pred in predictions.items():
    print("\\n" + "="*60)
    print(name)
    print(classification_report(y_test, pred, target_names=["Benign", "Malignant"], zero_division=0))

## 10. Compare classification metrics

In [ ]:
metric_cols = ["Accuracy", "Precision", "Recall", "F1-Score"]
results_df.set_index("Model")[metric_cols].plot(kind="bar", figsize=(14,6))
plt.title("Classification Metrics Comparison")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

results_df.set_index("Model")["Log Loss"].plot(kind="bar", figsize=(12,5))
plt.title("Log Loss Comparison (Lower is Better)")
plt.ylabel("Log Loss")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 11. Best model by metric

In [ ]:
for metric, direction in [("Accuracy", "max"), ("F1-Score", "max"), ("Recall", "max"), ("Log Loss", "min")]:
    idx = results_df[metric].idxmax() if direction == "max" else results_df[metric].idxmin()
    print(f"Best {metric}: {results_df.loc[idx, 'Model']} = {results_df.loc[idx, metric]:.4f}")
display(results_df.round(4))

## 12. Findings, suitability, and further improvement

- **Stacking:** a meta-classifier learns how to combine outputs from base models. Useful when base learners have complementary strengths; it is more complex and computationally expensive.
- **AdaBoost:** sequentially trains weak learners, focusing more on samples previously misclassified. Useful for structured data, but can be sensitive to noisy labels and outliers.
- **Majority Voting:** selects the class receiving the most votes. Simple and interpretable; works best when constituent models are reasonably accurate and make different errors.
- **Base models:** provide a baseline to verify whether ensembling improves results. An ensemble is not guaranteed to outperform every base model.
- **Medical interpretation:** inspect malignant-class recall and false negatives, not accuracy alone.

**Further research:** perform cross-validation and hyperparameter tuning, test on an independent dataset, investigate class imbalance and probability calibration, and compare execution time and model complexity.

**Note:** labels use the Kaggle dataset mapping: B = 0 (Benign), M = 1 (Malignant). For hard voting, Log Loss uses the fraction of base-estimator votes for class 1 as an estimated probability.

## 13. Conclusion

In [ ]:
print("The base classifiers and Stacking, AdaBoost, and majority Voting ensembles were evaluated.")
print("Use the metrics table, confusion matrices, and graphs above to support the final comparison.")
display(results_df.round(4))

## Screenshots to capture for the practical report
1. Dataset shape and class counts
2. Base-model training output
3. Stacking, AdaBoost, and Voting training output
4. Metric comparison table
5. Confusion matrices
6. Classification reports
7. Classification metric comparison graph
8. Log Loss graph
9. Best-model summary and conclusion